# 🏢 Employee Attrition & Workforce Analytics
### SyntecxHub Internship Project | IBM HR Analytics Dataset
---
> **Analyst:** Data Analytics Intern  
> **Organization:** SyntecxHub  
> **Dataset:** IBM HR Analytics Employee Attrition & Performance (1,470 records)  
> **Tools:** Python · Pandas · Matplotlib · Seaborn


## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Corporate color palette
NAVY  = '#0F172A'
TEAL  = '#0D9488'
CORAL = '#F43F5E'
SLATE = '#64748B'
WHITE = '#FFFFFF'
MGREY = '#E2E8F0'

plt.rcParams.update({
    'figure.facecolor': NAVY,  'axes.facecolor': NAVY,
    'axes.edgecolor': SLATE,   'axes.labelcolor': WHITE,
    'axes.titlecolor': WHITE,  'axes.titlesize': 13,
    'axes.titleweight': 'bold','xtick.color': MGREY,
    'ytick.color': MGREY,      'text.color': WHITE,
    'grid.color': '#1E293B',   'legend.facecolor': '#1E293B',
    'legend.edgecolor': SLATE, 'legend.labelcolor': WHITE,
    'font.family': 'DejaVu Sans', 'figure.dpi': 120,
})

print('Libraries loaded ✓')

In [ ]:
# Load cleaned dataset
df = pd.read_csv('../data/cleaned_employee_data.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

## 2. Dataset Overview & Validation

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Total Employees  : {len(df):,}')
print(f'Total Columns    : {df.shape[1]}')
print(f'Missing Values   : {df.isnull().sum().sum()}')
print(f'Duplicate Rows   : {df.duplicated().sum()}')
print()
attrition_counts = df['Attrition'].value_counts()
attrition_rate   = attrition_counts['Yes'] / len(df) * 100
print(f'Attrition Count  : {attrition_counts["Yes"]} ({attrition_rate:.1f}%)')
print(f'Retention Count  : {attrition_counts["No"]} ({100-attrition_rate:.1f}%)')
print()
print('Age Group Distribution:')
print(df['Age Group'].value_counts().sort_index())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Attrition by Department

In [ ]:
dept_attr = (
    df.groupby(['Department', 'Attrition'])
    .size().unstack(fill_value=0)
    .assign(Rate=lambda x: x['Yes'] / (x['Yes'] + x['No']) * 100)
    .sort_values('Rate', ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(dept_attr)); w = 0.35
ax.bar(x - w/2, dept_attr['No'],  w, label='Retained', color=TEAL,  zorder=3)
ax.bar(x + w/2, dept_attr['Yes'], w, label='Attrited', color=CORAL, zorder=3)
for i, (_, row) in enumerate(dept_attr.iterrows()):
    ax.text(i + w/2, row['Yes'] + 3, f'{row["Rate"]:.1f}%', ha='center', fontsize=10, color=CORAL, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(dept_attr.index, fontsize=11)
ax.set(xlabel='Department', ylabel='Employee Count', title='Attrition by Department')
ax.legend(); ax.yaxis.grid(True, zorder=0)
plt.tight_layout(); plt.show()

print('\n📊 INSIGHT: Human Resources has the highest attrition rate at ~19%, followed by Sales at ~20.6%.')
print('   R&D has the most employees but benefits from structured career pathways.')
print('   RECOMMENDATION: Implement targeted retention programs in Sales & HR immediately.')

### 3.2 Attrition by Job Role

In [ ]:
role_rate = (
    df.groupby('JobRole')['Attrition']
    .apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
    .sort_values(ascending=True)
)
colors = [CORAL if v >= 20 else TEAL for v in role_rate.values]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(role_rate.index, role_rate.values, color=colors, height=0.6, zorder=3)
for bar, val in zip(bars, role_rate.values):
    ax.text(val + 0.4, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10, color=WHITE, fontweight='bold')
ax.axvline(role_rate.mean(), color=SLATE, linestyle='--', linewidth=1.2, label=f'Avg {role_rate.mean():.1f}%')
ax.set(xlabel='Attrition Rate (%)', title='Attrition Rate by Job Role')
ax.xaxis.grid(True, zorder=0); ax.legend()
plt.tight_layout(); plt.show()

print('\n📊 INSIGHT: Sales Representative (~39.8%) and Laboratory Technician (~23.9%) are critical risk roles.')
print('   Managers and Research Directors show lowest attrition (<5%).')
print('   RECOMMENDATION: Review compensation bands and career ladders for Sales Representatives urgently.')

### 3.3 Annual Salary vs Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
groups = [df[df['Attrition']=='No']['Salary'].values, df[df['Attrition']=='Yes']['Salary'].values]
bp = ax.boxplot(groups, patch_artist=True, widths=0.45,
                medianprops=dict(color=WHITE, linewidth=2.5),
                whiskerprops=dict(color=MGREY), capprops=dict(color=MGREY),
                flierprops=dict(marker='o', color=SLATE, markersize=3))
bp['boxes'][0].set_facecolor(TEAL); bp['boxes'][1].set_facecolor(CORAL)
for i, (grp, lbl) in enumerate(zip(groups, ['Retained','Attrited']), 1):
    med = np.median(grp)
    ax.text(i, med+2500, f'${med:,.0f}', ha='center', fontsize=10, color=WHITE, fontweight='bold')
ax.set_xticklabels(['Retained','Attrited'], fontsize=12)
ax.set(ylabel='Annual Salary (USD)', title='Annual Salary Distribution vs Attrition')
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.yaxis.grid(True, zorder=0); plt.tight_layout(); plt.show()

retained_med = np.median(df[df['Attrition']=='No']['Salary'])
attrited_med = np.median(df[df['Attrition']=='Yes']['Salary'])
print(f'\n📊 INSIGHT: Retained employees median salary: ${retained_med:,.0f} vs Attrited: ${attrited_med:,.0f}')
print(f'   Gap of ${retained_med - attrited_med:,.0f} annually signals compensation is a primary attrition driver.')
print('   RECOMMENDATION: Benchmark entry-level salaries against market rates. Target 15-20% increase for at-risk bands.')

### 3.4 Work-Life Balance vs Attrition

In [ ]:
wlb = df.groupby(['WorkLifeBalance','Attrition']).size().unstack(fill_value=0)
wlb_pct = wlb.div(wlb.sum(axis=1), axis=0) * 100
labels = {1:'Poor (1)', 2:'Fair (2)', 3:'Good (3)', 4:'Excellent (4)'}
wlb_pct.index = [labels.get(i, str(i)) for i in wlb_pct.index]

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(wlb_pct))
for col, color, lbl in zip(['No','Yes'],[TEAL,CORAL],['Retained','Attrited']):
    vals = wlb_pct[col].values
    bars = ax.bar(wlb_pct.index, vals, bottom=bottom, color=color, label=lbl, zorder=3)
    for bar, val in zip(bars, vals):
        if val > 4:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+val/2, f'{val:.1f}%',
                    ha='center', va='center', fontsize=9, color=WHITE, fontweight='bold')
    bottom += vals
ax.set(xlabel='Work-Life Balance Rating', ylabel='% of Employees', title='Work-Life Balance vs Attrition')
ax.legend(); ax.yaxis.grid(True, zorder=0); ax.set_ylim(0,110)
plt.tight_layout(); plt.show()

print('\n📊 INSIGHT: Employees with Poor (1) work-life balance show the highest attrition percentage.')
print('   Good (3) rating has the largest population but still shows ~14% attrition.')
print('   RECOMMENDATION: Introduce flexible work arrangements and enforce minimum PTO utilization policies.')

### 3.5 Gender vs Attrition

In [ ]:
gender_rate  = df.groupby('Gender')['Attrition'].apply(lambda x: (x=='Yes').sum()/len(x)*100)
gender_count = df.groupby('Gender')['Attrition'].count()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(gender_rate.index, gender_rate.values, color=[TEAL,'#A78BFA'], width=0.4, zorder=3)
for bar, val, cnt in zip(bars, gender_rate.values, gender_count.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5, f'{val:.1f}%\n(n={cnt:,})',
            ha='center', va='bottom', fontsize=11, color=WHITE, fontweight='bold')
ax.set(xlabel='Gender', ylabel='Attrition Rate (%)', title='Attrition Rate by Gender')
ax.set_ylim(0, 25); ax.yaxis.grid(True, zorder=0)
plt.tight_layout(); plt.show()

print('\n📊 INSIGHT: Male employees show marginally higher attrition (~17%) vs Female (~14.8%).')
print('   The difference is not extreme — gender is not a primary driver but warrants monitoring.')
print('   RECOMMENDATION: Ensure equitable compensation and advancement opportunities across genders.')

### 3.6 Overtime vs Attrition

In [ ]:
ot = df.groupby(['OverTime','Attrition']).size().unstack(fill_value=0)
ot_rate = (ot['Yes'] / (ot['Yes']+ot['No']) * 100).reset_index()
ot_rate.columns = ['OverTime','Rate']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(ot_rate['OverTime'], ot_rate['Rate'], color=[TEAL,CORAL], width=0.4, zorder=3)
for bar, row in zip(bars, ot_rate.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, row.Rate+0.5, f'{row.Rate:.1f}%',
            ha='center', va='bottom', fontsize=14, color=WHITE, fontweight='bold')
ax.set(xlabel='Overtime Status', ylabel='Attrition Rate (%)', title='Overtime vs Attrition Rate')
ax.set_ylim(0, 40); ax.yaxis.grid(True, zorder=0)
plt.tight_layout(); plt.show()

no_r  = ot_rate[ot_rate['OverTime']=='No']['Rate'].values[0]
yes_r = ot_rate[ot_rate['OverTime']=='Yes']['Rate'].values[0]
print(f'\n📊 INSIGHT: Overtime employees attrition = {yes_r:.1f}% vs No Overtime = {no_r:.1f}%')
print(f'   This is a {yes_r/no_r:.1f}x multiplier — strongest single predictor in the dataset.')
print('   RECOMMENDATION: Cap overtime hours, implement TOIL policy, conduct workload audits in Sales & R&D.')

### 3.7 Attrition by Age Group

In [ ]:
order = ['18-25','26-35','36-45','46-55','55+']
age_rate  = df.groupby('Age Group')['Attrition'].apply(lambda x: (x=='Yes').sum()/len(x)*100).reindex(order)
age_count = df.groupby('Age Group').size().reindex(order)
colors    = [CORAL if v >= 25 else TEAL for v in age_rate.values]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(age_rate.index, age_rate.values, color=colors, width=0.5, zorder=3)
for bar, val, cnt in zip(bars, age_rate.values, age_count.values):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5, f'{val:.1f}%\n(n={cnt})',
            ha='center', va='bottom', fontsize=10, color=WHITE, fontweight='bold')
ax.axhline(age_rate.mean(), color=SLATE, linestyle='--', linewidth=1.2, label=f'Avg {age_rate.mean():.1f}%')
ax.set(xlabel='Age Group', ylabel='Attrition Rate (%)', title='Attrition Rate by Age Group')
ax.set_ylim(0, 50); ax.legend(); ax.yaxis.grid(True, zorder=0)
plt.tight_layout(); plt.show()

print(f'\n📊 INSIGHT: 18-25 cohort shows {age_rate["18-25"]:.1f}% attrition — highest of all groups.')
print('   26-35 is the largest population but also elevated due to career mobility.')
print('   RECOMMENDATION: Structured onboarding + mentoring programs for 18-25. Fast-track promotions for 26-35.')

### 3.8 Correlation Heatmap

In [ ]:
df_num = df.copy()
df_num['Attrition_Num'] = (df_num['Attrition']=='Yes').astype(int)
df_num['OverTime_Num']  = (df_num['OverTime']=='Yes').astype(int)
corr_cols = ['Attrition_Num','Age','DistanceFromHome','Education','EnvironmentSatisfaction',
             'JobInvolvement','JobLevel','JobSatisfaction','MonthlyIncome','NumCompaniesWorked',
             'OverTime_Num','RelationshipSatisfaction','StockOptionLevel','Experience',
             'WorkLifeBalance','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager']
corr = df_num[corr_cols].corr()
corr.index = corr.columns = [c.replace('_Num','') for c in corr.columns]

fig, ax = plt.subplots(figsize=(14, 10))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap=cmap, linewidths=0.4, linecolor='#1E293B',
            annot_kws={'size':7.5,'color':WHITE}, ax=ax, cbar_kws={'shrink':0.7})
ax.set_title('Feature Correlation Heatmap', pad=16)
ax.tick_params(axis='x', labelrotation=45, labelsize=9)
ax.tick_params(axis='y', labelrotation=0,  labelsize=9)
plt.tight_layout(); plt.show()

attrition_corr = corr['Attrition'].drop('Attrition').abs().sort_values(ascending=False)
print('\n📊 TOP CORRELATORS WITH ATTRITION:')
print(attrition_corr.head(8).to_string())

## 4. Summary of Key Findings

| # | Finding | Impact |
|---|---------|--------|
| 1 | **Overtime = 3x attrition risk** | Critical |
| 2 | **Sales Rep ~40% attrition rate** | Critical |
| 3 | **Salary gap: $62K retained vs $38K attrited** | High |
| 4 | **18-25 age group highest churn** | High |
| 5 | **Poor work-life balance drives exits** | Medium |
| 6 | **Sales & HR departments at risk** | Medium |